# Level 1 + 3 + 4: 
* Cài đặt cơ bản Eclat
* cài đặt tối ưu bằng cách sử dụng Bit Array và toán tử AND của Julia
* Đọc và xuất output SPMF

In [1]:
using Pkg
Pkg.activate("..")

# Nạp mã nguồn
include("../src/structures.jl")
include("../src/utils.jl")
include("../src/algorithm/eclat_optimized.jl")
include("../src/algorithm/eclat_basic.jl")

using ..Structures, .Utils, .EclatOptimized, .EclatBasic
println("Đã nạp môi trường và cả hai phiên bản thuật toán Eclat.")

  Activating project at `d:\Github\ai biet\Lab2-DM`


Đã nạp môi trường và cả hai phiên bản thuật toán Eclat.


# --- LỰA CHỌN BENCHMARK ---
 1. "../data/toy/toy_data.txt"      (min_sup gợi ý: 3)
 2. "../data/benchmark/mushroom.txt" (min_sup gợi ý: 2000)
 3. "../data/benchmark/chess.txt"    (min_sup gợi ý: 2500)

In [2]:
data_path = "../data/benchmark/accident.txt" 
min_sup = 199994

199994

# --- LỰA CHỌN THUẬT TOÁN ---
* true  => Chạy bản Tối ưu (BitArray)
* false => Chạy bản Cơ bản (Set)


In [3]:
# --- LỰA CHỌN THUẬT TOÁN ---
# true  => Chạy bản Tối ưu (BitArray)
# false => Chạy bản Cơ bản (Set)
use_optimized = true 

true

In [4]:
println("File dữ liệu: ", data_path)
println("Ngưỡng min_sup: ", min_sup)
println("Thuật toán đang chọn: ", use_optimized ? "TỐI ƯU (BitArray)" : "CƠ BẢN (Set)")

# Đọc dữ liệu thô
tidsets, n_trans = Utils.read_spmf(data_path)

File dữ liệu: ../data/benchmark/accident.txt
Ngưỡng min_sup: 199994
Thuật toán đang chọn: TỐI ƯU (BitArray)


(Dict{Int64, BitVector}(56 => [0, 0, 0, 1, 1, 0, 1, 0, 1, 0  …  0, 0, 1, 0, 0, 0, 1, 1, 0, 1], 35 => [0, 1, 0, 0, 0, 1, 1, 0, 0, 0  …  1, 1, 0, 1, 0, 0, 0, 0, 0, 0], 425 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 429 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 60 => [0, 0, 0, 1, 0, 0, 1, 1, 0, 1  …  0, 1, 1, 1, 0, 0, 1, 1, 0, 0], 220 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 308 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 67 => [0, 0, 0, 0, 1, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 1, 0, 0, 0], 215 => [0, 0, 0, 0, 0, 0, 0, 0, 0, 0  …  0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 73 => [0, 0, 0, 0, 0, 1, 0, 0, 0, 0  …  0, 0, 0, 0, 1, 0, 0, 0, 0, 0]…), 340183)

## Nếu cell bên dưới bị lỗi 'ItemsetBasic' not defined in `Main` thì restart kernel và chạy lại

In [5]:
results_basic = Vector{ItemsetBasic}()
results_optimized = Vector{ItemsetOptimized}()


if use_optimized
    # 1. Chuẩn bị dữ liệu cho bản Tối ưu
    P = [ItemsetOptimized([item], tidsets[item]) for item in sort(collect(keys(tidsets)))]
    filter!(x -> x.support >= min_sup, P)
    
    println("Đang chạy Eclat TỐI ƯU...")
    @time run_eclat_optimized(P, min_sup, results_optimized)
    
    final_results = results_optimized
else
    # 2. Chuẩn bị dữ liệu cho bản Cơ bản (Chuyển BitArray sang Set)
    P_basic = [ItemsetBasic([item], Set(findall(tidsets[item]))) for item in sort(collect(keys(tidsets)))]
    filter!(x -> x.support >= min_sup, P_basic)
    
    println("Đang chạy Eclat CƠ BẢN...")
    @time run_eclat_basic(P_basic, min_sup, results_basic)
    
    final_results = results_basic
end

println("\n HOÀN TẤT!")
println("Số lượng tập phổ biến tìm được: ", length(final_results))

Đang chạy Eclat TỐI ƯU...
  0.951622 seconds (887.18 k allocations: 263.158 MiB, 31.07% gc time, 50.43% compilation time)

 HOÀN TẤT!
Số lượng tập phổ biến tìm được: 2451


In [6]:
# Tên file xuất ra
spmf_output_file = "output_spmf.txt"

# Gọi hàm write_results từ module Utils
Utils.write_results(final_results, spmf_output_file)

println("Đã xuất kết quả theo chuẩn SPMF ra file: $spmf_output_file")

Đã xuất kết quả theo chuẩn SPMF ra file: output_spmf.txt


In [7]:
println("Một số kết quả tìm được:")
println("-"^40)
for i in 1:min(10, length(final_results))
    r = final_results[i]
    println("Tập: $(r.items) | Support: $(r.support)")
end
println("-"^40)

Một số kết quả tìm được:
----------------------------------------
Tập: [1] | Support: 234704
Tập: [1, 12] | Support: 234461
Tập: [1, 12, 16] | Support: 230754
Tập: [1, 12, 16, 17] | Support: 230737
Tập: [1, 12, 16, 17, 18] | Support: 230327
Tập: [1, 12, 16, 17, 18, 21] | Support: 203615
Tập: [1, 12, 16, 17, 18, 31] | Support: 222374
Tập: [1, 12, 16, 17, 18, 43] | Support: 200047
Tập: [1, 12, 16, 17, 21] | Support: 203991
Tập: [1, 12, 16, 17, 29] | Support: 200197
----------------------------------------


# Phần 2: Tái tạo đúng kết quả (Unit test)

* Dùng lệnh này trong terminal:  julia --project=. test/runtests.jl